## Google Colab Classification Notebook
This notebook runs classification experiments from within Google Colab.

In [ ]:
# Initial imports
import sys
import os
from google.colab import drive

In [ ]:
# Clone the repository to the Colab environment
!git clone https://github.com/gemixin/touch-ex

In [ ]:
# Set up the repository path and add it to the Python path
repo_path = "/content/touch-ex"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
os.chdir(repo_path)

In [ ]:
# Mount Google Drive so experiment outputs persist after the Colab session ends
drive.mount("/content/drive")

# Set up directories for experiment outputs
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/touch-ex"
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# Install the project dependencies. Colab's existing CUDA-enabled PyTorch is used
# when it already satisfies the torch and torchvision requirements
%pip install -q -r requirements.txt

In [ ]:
from models.experiments import classify

# --- Configurable parameters ---

# Model types to compare
# Choose from 'baseline', 'resnet18', 'efficientnet_b0', 'vit_b_16', 'deit_tiny', or
# 't3_tiny'
"""
MODEL_TYPES = [
    "baseline",
    "resnet18",
    "efficientnet_b0",
    "vit_b_16",
    "deit_tiny",
    "t3_tiny",
]
"""
MODEL_TYPES = ["vit_b_16"]

# Target label for classification
# Choose from 'object', 'object_region', 'force_level', or 'motion'
TARGET_LABEL = "object"

# Experiment name for tracking results
# EXPERIMENT_NAME = "finetuned_comparison"
EXPERIMENT_NAME = "quick_vit_test"

# Randomisation settings
SEED = 129
DETERMINISTIC = True

# True trains only the classification head of pretrained models
# Baseline models are always trained end-to-end
FREEZE_BACKBONE = False

# Values here override keys in provided data_config and train_config files
DATA_CONFIG_OVERRIDES = {
    "num_workers": 8,
}
TRAIN_CONFIG_OVERRIDES = {
    "num_epochs": 1,
}

# t-SNE feature plot settings
PLOT_TSNE = True
TSNE_MAX_SAMPLES = -1

# Paths for files and directories
DATA_CONFIG_PATH = os.path.join(repo_path, "configs/default_data_config.json")
TRAIN_CONFIG_PATH = os.path.join(
    repo_path,
    "configs/frozen_train_config.json"
    if FREEZE_BACKBONE
    else "configs/finetuned_train_config.json",
)
BASELINE_TRAIN_CONFIG_PATH = os.path.join(repo_path, "configs/baseline_train_config.json")

# --- Train and evaluate models ---

models, histories, test_results = classify(
    model_types=MODEL_TYPES,
    target_label=TARGET_LABEL,
    experiment_name=EXPERIMENT_NAME,
    seed=SEED,
    deterministic=DETERMINISTIC,
    freeze_backbone=FREEZE_BACKBONE,
    data_config_overrides=DATA_CONFIG_OVERRIDES,
    train_config_overrides=TRAIN_CONFIG_OVERRIDES,
    plot_tsne=PLOT_TSNE,
    tsne_max_samples=TSNE_MAX_SAMPLES,
    data_config_path=DATA_CONFIG_PATH,
    train_config_path=TRAIN_CONFIG_PATH,
    baseline_train_config_path=BASELINE_TRAIN_CONFIG_PATH,
    results_dir=RESULTS_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
)